# Lab 6.3: Naming Customer Segments with a Surrogate Tree

Run the cells from top to bottom. You will cluster customers of an online retailer into four segments, explain the segments with small decision trees, and measure how faithful each explanation is, segment by segment.

Data: Online Retail II (Chen, UCI Machine Learning Repository, doi 10.24432/C5CG6D, license CC BY 4.0). We use the sheet for December 2010 to December 2011. The notebook downloads the 45 MB zip file once into a temporary folder; you need `openpyxl` to read the spreadsheet.

## 1. From invoice lines to one row per customer

In [1]:
import tempfile, urllib.request, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

TMP = Path(tempfile.gettempdir()) / "xai-ch06"; TMP.mkdir(exist_ok=True)
xl = TMP / "online_retail_II.xlsx"
if not xl.exists():
    z = TMP / "or2.zip"
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip", z)
    zipfile.ZipFile(z).extractall(TMP)
d = pd.read_excel(xl, sheet_name="Year 2010-2011")
print("invoice lines:", len(d))
d = d.dropna(subset=["Customer ID"])
d = d[(d.Quantity > 0) & (d.Price > 0) & (~d.Invoice.astype(str).str.startswith("C"))]
d["Amount"] = d.Quantity * d.Price
ref = d.InvoiceDate.max() + pd.Timedelta(days=1)
g = d.groupby("Customer ID")
c = pd.DataFrame({"recency_days": (ref - g.InvoiceDate.max()).dt.days,     # days since last purchase
                  "tenure_days": (ref - g.InvoiceDate.min()).dt.days,      # days since first purchase
                  "frequency": g.Invoice.nunique(),                        # number of invoices
                  "volume": g.Amount.sum()})                               # total spend
print("lines kept:", len(d), " customers:", len(c))
c.describe().round(1)

invoice lines: 541910


lines kept: 397885  customers: 4338


,recency_days,tenure_days,frequency,volume
count,4338.0,4338.0,4338.0,4338.0
mean,92.5,223.3,4.3,2054.3
std,100.0,117.9,7.7,8989.2
min,1.0,1.0,1.0,3.8
25%,18.0,113.0,1.0,307.4
50%,51.0,249.0,2.0,674.5
75%,142.0,327.0,5.0,1661.7
max,374.0,374.0,209.0,280206.0


**Checkpoint 1.** 541,910 invoice lines in the sheet, 397,885 kept after removing lines without a customer, returns and zero prices, and 4,338 customers. Half of the customers bought at most twice.

## 2. Cluster the customers

In [2]:
Z = StandardScaler().fit_transform(np.log1p(c))
lab = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Z).labels_
print("segment sizes:", np.bincount(lab).tolist())
c.assign(segment=lab).groupby("segment").median().round(0)

segment sizes: [652, 1332, 1423, 931]


,recency_days,tenure_days,frequency,volume
segment,,,,
0,8.0,359.0,11.0,3983.0
1,212.0,261.0,1.0,307.0
2,36.0,276.0,4.0,1197.0
3,33.0,49.0,1.0,352.0


**Checkpoint 2.** Four segments of 652, 1,332, 1,423 and 931 customers. Look at the medians and give each segment a name *before* you build any tree. The clustering itself only returns the numbers 0 to 3.

## 3. Explain the segments with trees of depth 2, 3 and 4

In [3]:
Xtr, Xte, ytr, yte = train_test_split(c, lab, test_size=0.3, random_state=0, stratify=lab)
for depth in (1, 2, 3, 4, 5):
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(Xtr, ytr)
    p = t.predict(Xte)
    per = [(p[yte == s] == s).mean() for s in range(4)]
    print("depth %d: overall fidelity %.3f, per segment %s, leaves %d" % (depth, (p == yte).mean(), np.round(per, 2).tolist(), t.get_n_leaves()))

depth 1: overall fidelity 0.554, per segment [0.0, 0.87, 0.87, 0.0], leaves 2
depth 2: overall fidelity 0.761, per segment [0.0, 0.82, 0.93, 0.96], leaves 4
depth 3: overall fidelity 0.879, per segment [0.88, 0.85, 0.86, 0.96], leaves 8
depth 4: overall fidelity 0.896, per segment [0.8, 0.93, 0.88, 0.94], leaves 16
depth 5: overall fidelity 0.919, per segment [0.9, 0.92, 0.91, 0.94], leaves 29


**Checkpoint 3.** Depth 2 reaches an overall fidelity of about 0.76 but assigns none of segment 0, the most valuable customers, correctly: a four-leaf tree has no room left for them. Depth 3 reaches about 0.88 and describes all four segments at 0.85 or better.

In [4]:
t3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
print(export_text(t3, feature_names=list(c.columns), decimals=0))

|--- recency_days <= 104
|   |--- tenure_days <= 90
|   |   |--- volume <= 1307
|   |   |   |--- class: 3
|   |   |--- volume >  1307
|   |   |   |--- class: 3
|   |--- tenure_days >  90
|   |   |--- frequency <= 6
|   |   |   |--- class: 2
|   |   |--- frequency >  6
|   |   |   |--- class: 0
|--- recency_days >  104
|   |--- volume <= 1040
|   |   |--- frequency <= 4
|   |   |   |--- class: 1
|   |   |--- frequency >  4
|   |   |   |--- class: 2
|   |--- volume >  1040
|   |   |--- frequency <= 2
|   |   |   |--- class: 1
|   |   |--- frequency >  2
|   |   |   |--- class: 2



**Checkpoint 4.** The top split is on recency (about 104 days), then tenure and frequency. Write each segment's description as one sentence a marketing manager could read, using only the rules of the tree, and put the per-segment fidelity next to it.

## 4. Optional: how stable are the names?

Rerun Section 2 with `random_state=1` and `random_state=2` for KMeans, and Section 3 again. Do the segments keep their medians? Does the tree keep its first split? A name that changes with the random seed is a name for the algorithm's run, not for the customers.